In [ ]:
!pip install -q setfit sentence-transformers transformers datasets pandas scikit-learn emoji contractions kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Local module imports (ensure dataset.py and preprocessing.py are in your workspace)
from dataset import load_splits, load_preprocessed, save_preprocessed
from preprocessing import preprocess_batch

# 2. Custom PyTorch Dataset
class IMDBDataset(torch.utils.data.Dataset):
    """
    Custom PyTorch Dataset class for IMDB text classification.
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 3. Metrics Calculator
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 4. Preprocessing Helper with Disk Caching (Similar to train_sbert.py)
def get_preprocessed_texts(df_train, df_val, df_test):
    result = []
    for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
        cached = load_preprocessed(split_name)
        if cached is not None:
            result.append(cached)
        else:
            print(f"Preprocessing {split_name} ({len(df):,} reviews)…")
            texts = preprocess_batch(df["review"].tolist())
            save_preprocessed(texts, split_name)
            result.append(texts)
    return tuple(result)

# ==========================================
# Main Executable Flow for Notebook Cell
# ==========================================

# 1. Enforce device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Hardware device confirmed: NVIDIA CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Hardware device confirmed: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Hardware device confirmed: CPU (Note: Training will be slow without a GPU)")

# 2. Setup output directory
output_dir = "./output_PreProcessed"
checkpoint_dir = "./checkpoints/sentencebert_classifier"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# 3. Configuration Parameters
model_name = "sentence-transformers/all-MiniLM-L6-v2"
max_length = 128
batch_size = 16
epochs = 3
random_seed = 42

# 4. Load Data using dataset.py
print("Loading dataset splits...")
df_train, df_val, df_test = load_splits(
    seed=random_seed,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

# Output summary
print("\n========== Dataset Split Summary ==========")
print(f"Training records (70%): {len(df_train)}")
print(f"Validation records (15%): {len(df_val)}")
print(f"Testing records (15%): {len(df_test)}")
print("===========================================\n")

# 5. Preprocess texts using preprocessing.py
print("Preprocessing reviews...")
train_texts, val_texts, test_texts = get_preprocessed_texts(df_train, df_val, df_test)

train_labels = df_train["label"].tolist()
val_labels = df_val["label"].tolist()

# 6. Tokenize text data
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizing text data...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=max_length)

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)

# 7. Initialize model
print("Initializing model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 8. Training configuration and trainer initialization
training_args = TrainingArguments(
    output_dir=os.path.join(output_dir, "sentencebert_training_results"),
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=(device.type == "cuda"), # Set to True only when running on CUDA GPU
    dataloader_num_workers=0,
    gradient_accumulation_steps=1,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 9. Train model
print("Beginning training...")
trainer.train()

# 10. Save fine-tuned model and tokenizer
print(f"\nSaving fine-tuned model and tokenizer to {checkpoint_dir}...")
trainer.save_model(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)

print("\nProcess finished successfully. Model saved.")


Hardware device confirmed: NVIDIA CUDA
Loading dataset splits...
[dataset] Splits — train: 34,999  val: 7,501  test: 7,500

========== Dataset Split Summary ==========
Training records (70%): 34999
Validation records (15%): 7501
Testing records (15%): 7500

Preprocessing reviews...
Preprocessing train (34,999 reviews)…
[dataset] Saved preprocessed train texts → /content/data/preprocessed_train.pkl
Preprocessing val (7,501 reviews)…
[dataset] Saved preprocessed val texts → /content/data/preprocessed_val.pkl
Preprocessing test (7,500 reviews)…
[dataset] Saved preprocessed test texts → /content/data/preprocessed_test.pkl
Loading tokenizer for sentence-transformers/all-MiniLM-L6-v2...
Tokenizing text data...
Initializing model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Beginning training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.325800,0.342280,0.856552,0.865969,0.812529,0.926933
2,0.266291,0.297874,0.881482,0.881609,0.880553,0.882667
3,0.215009,0.334901,0.880816,0.883503,0.863914,0.904000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Saving fine-tuned model and tokenizer to ./checkpoints/sentencebert_classifier...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Process finished successfully. Model saved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')